# ND1 M7 — Quickstart

본 노트북은 5분 안에 핵심 기능을 둘러보는 예제입니다.

## 학습 순서
1. 기본 도구 (회전 행렬·DH 변환)
2. 정기구학 (FK) — 관절각 → 말단부
3. 해석적 IK — 코사인 법칙으로 두 해
4. 수치 IK — DLS 알고리즘
5. 야코비안 + 조작성

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.getcwd()))   # 상위 폴더 import 가능

import numpy as np
import matplotlib.pyplot as plt

## 1. 기본 도구

In [ ]:
from src.robot_helpers import Rz, dh_matrix

# Rz(90°) — x축을 y축으로
R = Rz(np.pi / 2)
print('Rz(90°):')
print(R.round(3))

# DH 단순 케이스 (a=0.3, 나머지 0)
T = dh_matrix(0.3, 0, 0, 0)
print('\nDH (a=0.3):')
print(T.round(3))

## 2. 정기구학 (FK)

In [ ]:
from src.robot_arm import RobotArm3DOF

robot = RobotArm3DOF()
print(f'최대 도달 거리: {robot.reach_max:.2f} m')

# 자세 1: 일직선
pos, T = robot.fk([0, 0, 0])
print(f'\n[0, 0, 0]: 말단부 = ({pos[-1][0]:.3f}, {pos[-1][1]:.3f})')

# 자세 2: 30°, 60°, 0°
pos, T = robot.fk([np.radians(30), np.radians(60), 0])
print(f'[30°, 60°, 0°]: 말단부 = ({pos[-1][0]:.3f}, {pos[-1][1]:.3f})')

# 시각화
fig, ax = plt.subplots(figsize=(7, 7))
robot.visualize([np.radians(30), np.radians(60), np.radians(15)],
                 ax=ax, show_workspace=True, title='3DOF 평면 로봇 자세')
plt.show()

## 3. 해석적 IK (2DOF, 코사인 법칙)

In [ ]:
from src.ik_analytical import ik_2dof

sols = ik_2dof(0.4, 0.2)
for label, (t1, t2) in zip(['elbow up  ', 'elbow down'], sols):
    print(f'{label}: θ1={np.degrees(t1):6.2f}°, θ2={np.degrees(t2):6.2f}°')

## 4. 수치 IK (DLS)

In [ ]:
from src.ik_numerical import ik_dls

theta, hist = ik_dls(robot, [0.6, 0.2])
print(f'수렴: {len(hist)}회, 최종 오차 = {hist[-1]:.2e} m')
print(f'관절각 (도): {np.degrees(theta).round(2)}')

# 수렴 곡선
fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogy(hist, 'o-', color='steelblue')
ax.set_xlabel('반복 횟수 k'); ax.set_ylabel('‖e‖ [m, log scale]')
ax.set_title('IK 수렴 곡선'); ax.grid(True, alpha=0.3)
plt.show()

## 5. 야코비안 + 조작성

In [ ]:
from src.jacobian import jacobian_analytical_3dof, manipulability

thetas = [np.radians(45), np.radians(60), np.radians(30)]
J = jacobian_analytical_3dof(thetas)
print('야코비안 J (2×3):')
print(J.round(3))
print(f'\n조작성 w = {manipulability(J):.4f}')

# 특이점 자세 비교
J_sing = jacobian_analytical_3dof([0, 0, 0])   # 완전 펴짐
print(f'특이점 w = {manipulability(J_sing):.2e}  (≈ 0)')

## 다음 단계

- `scripts/lab1_two_solutions.py` 실행 → 4가지 목표 + 두 해 시각화
- `scripts/lab2_pbl_main.py` 실행 → **PBL 핵심 산출물**
- `scripts/lab3_waypoints_manipulability.py` → 조작성 지형도

각 스크립트는 `results/` 폴더에 PNG/CSV를 자동 저장합니다.